# Energy Futures Forward Inflation Models Out-of-Sample

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
from   matplotlib.lines import Line2D
from   matplotlib.ticker import FuncFormatter


import statsmodels.api as sm
from   statsmodels.regression.rolling import RollingOLS

In [2]:
energy_tickers = ["CL", "CO", "HO", "NG", "QS", "XB"]
inf_tickers    = ["FWISBP55", "FWISUS55"]

In [23]:
data_path = r"A:\BBGData\data"
inf_paths = [os.path.join(data_path, ticker + ".parquet") for ticker in inf_tickers]
df_inf    = (pd.read_parquet(
    path = inf_paths, engine = "pyarrow").
    assign(country = lambda x: np.where(x.security.str.split(" ").str[0] == "FWISBP55", "UK", "US")))

In [24]:
fut_path   = r"A:\BBGFuturesManager_backup_backup\data\PXFront"
fut_paths  = [os.path.join(fut_path, ticker + ".parquet") for ticker in energy_tickers]
df_fut_rtn = (pd.read_parquet(
    path = fut_paths, engine = "pyarrow").
    assign(security = lambda x: x.security.str.split(" ").str[0]).
    pivot(index = "date", columns = "security", values = "PX_LAST").
    pct_change().
    reset_index().
    melt(id_vars = "date", var_name = "fut_ticker", value_name = "fut_rtn").
    dropna())

In [25]:
df_combined = (df_inf.pivot(
    index = "date", columns = "country", values = "value").
    apply(lambda x: np.log(x)).
    diff().
    shift().
    reset_index().
    melt(id_vars = "date", value_name = "inf_diff").
    dropna().
    merge(right = df_fut_rtn, how = "inner", on = ["date"]))

1. Beta Port
2. UK Decile OLS
3. US Decile OLS
4. Z-Score

# Beta Port

In [26]:
df_combined

,date,country,inf_diff,fut_ticker,fut_rtn
0,2004-04-30,UK,0.019608,CL1,0.001903
1,2004-04-30,UK,0.019608,CO1,0.002999
2,2004-04-30,UK,0.019608,HO1,0.002057
3,2004-04-30,UK,0.019608,NG1,-0.010466
4,2004-04-30,UK,0.019608,QS1,0.033461
...,...,...,...,...,...
59892,2024-10-28,US,-0.010759,CO1,-0.061219
59893,2024-10-28,US,-0.010759,HO1,-0.048936
59894,2024-10-28,US,-0.010759,NG1,-0.074062
59895,2024-10-28,US,-0.010759,QS1,-0.046950


In [35]:
def _get_oos_beta(df: pd.DataFrame) -> pd.DataFrame: 

    df_out = (RollingOLS(
        endog     = df.fut_rtn,
        exog      = sm.add_constant(df.inf_diff),
        expanding = True,
        min_nobs  = 30).
        fit().
        params.
        rename(columns = {
            "const"   : "alpha",
            "inf_diff": "beta"}).
        assign(lag_beta = lambda x: x.beta.shift()).
        merge(right = df, how = "inner", on = ["date"]))

    return df_out

df_oos_beta = (df_combined.assign(
    group_var = lambda x: x.country + " " + x.fut_ticker).
    set_index("date").
    groupby("group_var").
    apply(_get_oos_beta, include_groups = False).
    reset_index())

In [36]:
print(os.getcwd())

C:\Users\Diego\Documents\GitHub\CommodityForwardInflation
